# Aprenentatge no supervisat

Al contrari que el supervisat, el no supervisat treballa en datasets `no etiquetats`, és a dir sense `target`. 

L'objectiu és classificar les observacions, als distints grups els podem posar després, o no, etiquetes. 

Com a part del procés es poden detectar dades que no corresponen a cap classe. Això són anomalíes o `outliers`. De vegades els outliers molesten per al bon funcionament dels models, de vegades podem utilitzar aquests algorismes per detectar outliers per a altres algorismes supervisats i de vegades estarem fen els models perquè el que ens interessa són precisament detectar anomalies. 


Els sistemes de *clustering* funcionen avaluant la similitud o **afinitat** entre diferents observacions o respecte a un grup ja existent. Com major siga esta afinitat, més probable serà que s'integreguen en el mateix conjunt. El seu propòsit fonamental es dividix en dos objectius complementaris:

1. **Alta cohesió:** Aconseguir la màxima similitud possible entre els elements que formen part d'un mateix clúster.
2. **Baix acoblament:** Garantir la mínima relació o afinitat entre els components de clústers diferents.

La gran majoria d'estos algorismes comparteixen un caràcter **iteratiu** perquè s'enfoquen com a problemes d'optimització. El procés habitual seguix estos passos:

* Es partix d'una **solució inicial** (que pot ser completament aleatòria), assignant les observacions a diferents grups.
* Esta distribució inicial es va **refinant pas a pas** en successives iteracions.
* Alguns mètodes exigeixen definir el nombre de clústers per endavant, mentre que uns altres el poden modificar en temps d'execució (per exemple, fusionant grups).
* El procés finalitza quan s'arriba a un punt d'estabilitat —on l'assignació de les dades ja no canvia— o quan es complix un requisit mínim de qualitat.


## Tipus principals d'algorismes

### 1. K-Means (Basat en distàncies)

Cada clúster està definit per un **centroide** (el seu punt central). Cada dada s'assigna al grup del centroide que té més a prop. A continuació, es recalculen les coordenades dels centroides fent la mitjana aritmètica de totes les dades d'eixe clúster. El cicle es repeteix fins que cap dada canvia de grup. Requereix fixar prèviament el nombre de clústers ($k$) i triar uns centres inicials de manera arbitrària.

### 2. FCM: Fuzzy C-Means (Basat en lògica difusa)

És una evolució de K-Means. En lloc de vincular una dada a un únic clúster de forma absoluta, utilitza funcions matemàtiques per a calcular la **probabilitat de pertinença** a cada grup. Finalment, cada observació s'associa al clúster on esta funció de pertinença prenga el valor màxim.

### 3. DBSCAN (Basat en densitat)

Agrupa en un mateix clúster aquells punts situats en **zones d'alta concentració** de dades (amb molts veïns pròxims). Per contra, classifica com a valors atípics o *outliers* tots aquells punts situats en regions aïllades o de baixa densitat.

### 4. Clústering jeràrquic

Genera una estructura de grups en forma d'arbre que es pot visualitzar mitjançant un **dendrograma**, on la base o arrel és el conjunt complet de dades i els nivells inferiors formen particions cada vegada més específiques. Pot funcionar de dos formes:

* **Aglomeratiu (*bottom-up*):** Es comença considerant cada dada com un clúster independent i s'aniran unint progressivament de baix a dalt.
* **Divisiu (*top-down*):** Es comença amb un únic clúster que abasta tot el *dataset* i s'anirà dividint successivament de dalt a baix.

## K-Means

Aquest algorisme de `clustering` es basa en la distància. El que es fa es definir un `centroid` que és el punt central d'un cluster. Cada observació es assignada al cluster que té el centroid més proper. 

El centroid és la mitjana aritmètica de totes les observacions del cluster. 

La dificultat d'aquest algorisme és determinar els centroids:

- Es trien k dades com a centroids. 
- S'assigna cada dada al centroid més proper, per tant tenim k grups.
- Si en el pass anterior hi ha algun canvi de grup, el torna a obtenir el centroid de cada grup que ha canviat i es torna al pas anterior. 

https://www.unioviedo.es/compnum/labs/new/kmeans.html



In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from IPython.display import HTML

# 1. Generació de dades sintètiques en 2 dimensions
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=42)

# 2. Configuració prèvia de K-means per capturar els passos
k = 4
max_iter = 10

# Guardarem els centroides de cada iteració per animar-los
centroids_history = []
labels_history = []

# Inicialització aleatòria dels centroides
init_centroids = X[np.random.choice(X.shape[0], k, replace=False)]
current_centroids = init_centroids

for i in range(max_iter):
    # Assignar clústers basats en els centroides actuals
    kmeans = KMeans(n_clusters=k, init=current_centroids, n_init=1, max_iter=1, random_state=42)
    kmeans.fit(X)
    
    labels_history.append(kmeans.labels_)
    centroids_history.append(kmeans.cluster_centers_)
    
    # Comprovar si els centroides han deixat de moure'ixer (convergent)
    if np.allclose(current_centroids, kmeans.cluster_centers_):
        break
    current_centroids = kmeans.cluster_centers_

# 3. Configuració de la figura de Matplotlib
fig, ax = plt.subplots(figsize=(8, 6))
scat = ax.scatter(X[:, 0], X[:, 1], c=labels_history[0], cmap='viridis', s=50, alpha=0.7)
cent_scat = ax.scatter(centroids_history[0][:, 0], centroids_history[0][:, 1], 
                       c='red', marker='X', s=200, edgecolor='black', label='Centroides')

ax.set_title('Evolució de K-Means (Pas 0)', fontsize=14)
ax.set_xlabel('Característica 1')
ax.set_ylabel('Característica 2')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# 4. Funció d'actualització per a l'animació
def update(frame):
    # Actualitzar els colors dels punts segons l'assignació del clúster
    scat.set_array(labels_history[frame])
    
    # Actualitzar la posició dels centroides
    cent_scat.offsets = centroids_history[frame]
    
    ax.set_title(f'Evolució de K-Means - Iteració {frame + 1}', fontsize=14)
    return scat, cent_scat

# Crear l'animació
anim = FuncAnimation(fig, update, frames=len(centroids_history), interval=800, blit=False)

# Tancar la figura estàtica per evitar que es mostri doble al notebook
plt.close()

# 5. Mostrar l'animació directament al Jupyter Notebook
HTML(anim.to_jshtml())